# 12 Causal Inference — Reference Solutions

Complete solutions for the causal inference exercises based on the Songbai Nursing Home Legionella cluster investigation.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import statsmodels.formula.api as smf

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

## Question 1: Attributable Risk of Hydrotherapy Exposure

In [ ]:
# Hydrotherapy exposure
hydro_exp = df[df["hydrotherapy_use"] == 1]
hydro_unexp = df[df["hydrotherapy_use"] == 0]

risk_hydro_exp = hydro_exp["infected"].mean()
risk_hydro_unexp = hydro_unexp["infected"].mean()
risk_total = df["infected"].mean()

AR_hydro = risk_hydro_exp - risk_hydro_unexp
PAR_hydro = risk_total - risk_hydro_unexp
PAR_pct_hydro = PAR_hydro / risk_total * 100

print("=== Hydrotherapy Exposure ===")
print(f"Attack rate among users: {risk_hydro_exp:.1%}")
print(f"Attack rate among non-users: {risk_hydro_unexp:.1%}")
print(f"AR = {AR_hydro:.3f}")
print(f"PAR% = {PAR_pct_hydro:.1f}%")

# Shower exposure (for comparison)
shower_exp = df[df["shower_use"] == 1]
shower_unexp = df[df["shower_use"] == 0]
risk_sh_exp = shower_exp["infected"].mean()
risk_sh_unexp = shower_unexp["infected"].mean()
AR_shower = risk_sh_exp - risk_sh_unexp
PAR_shower = risk_total - risk_sh_unexp
PAR_pct_shower = PAR_shower / risk_total * 100

print(f"\n=== Shower Exposure (comparison) ===")
print(f"AR = {AR_shower:.3f}")
print(f"PAR% = {PAR_pct_shower:.1f}%")

print(f"\n=== Comparison ===")
if abs(AR_shower) > abs(AR_hydro):
    print("→ Shower exposure has the larger AR, contributing more to infection")
else:
    print("→ Hydrotherapy exposure has the larger AR")

print("\n→ AR represents 'the amount of risk that could be removed by eliminating the exposure, if the causal relationship holds'")
print("→ Assumptions: (1) the causal relationship holds (2) no confounding (3) the exposure is removable")

## Question 2: Changing the DiD Intervention Date

In [ ]:
cases = df[df["infected"] == 1].copy()
all_dates = pd.date_range("2026-01-12", "2026-01-28", freq="D")

# Treated group / control group
treated_mask = (cases["floor"].isin([2, 3])) & (cases["wing"] == "B")
treated_daily = cases[treated_mask].groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)
control_daily = cases[~treated_mask].groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# Compare different intervention dates
for cutoff in ["2026-01-22", "2026-01-25"]:
    panel = pd.DataFrame({
        "date": list(all_dates) * 2,
        "treated": [1] * len(all_dates) + [0] * len(all_dates),
        "daily_cases": list(treated_daily.values) + list(control_daily.values),
    })
    panel["post"] = (panel["date"] >= cutoff).astype(int)

    model = smf.ols("daily_cases ~ treated + post + treated:post", data=panel).fit()
    coef = model.params["treated:post"]
    pval = model.pvalues["treated:post"]

    print(f"Intervention date = {cutoff}: treated:post = {coef:.3f}, p = {pval:.4f}")

print("\n→ Changing the intervention date affects the DiD result")
print("→ Reason: the number of observation days before and after the intervention differs, and so does the distribution of cases")
print("→ The intervention date must be based on the actual event (disinfection really happened); it cannot be picked arbitrarily")
print("→ Cherry-picking an intervention date to produce a significant result = p-hacking")

## Question 3 (Challenge): Demonstrating Collider Bias

In [ ]:
from epi_learning import risk_ratio

# RR for the whole sample
ct_all = pd.crosstab(df["shower_use"], df["infected"])
rr_all = risk_ratio(ct_all.iloc[1, 1], ct_all.iloc[1].sum(), ct_all.iloc[0, 1], ct_all.iloc[0].sum())
print(f"=== Whole-sample RR (shower → infected) ===")
print(f"RR = {rr_all:.3f}")

# Restrict to hospitalized patients
hosp = df[df["hospitalized"] == 1].copy()
print(f"\nHospitalized patients: {len(hosp)}")
print(f"shower_use distribution among hospitalized: {hosp['shower_use'].value_counts().to_dict()}")
print(f"infected distribution among hospitalized: {hosp['infected'].value_counts().to_dict()}")

# Are all hospitalized patients infected?
if hosp["infected"].nunique() == 1:
    print("\n→ All hospitalized patients are infected (infected=1), so RR cannot be computed")
    print("→ This is precisely the extreme case of collider bias!")
    print("→ Because only people who are infected and severe get hospitalized")
    print("→ Among hospitalized patients, the relationship between shower_use and infected is distorted")
else:
    ct_hosp = pd.crosstab(hosp["shower_use"], hosp["infected"])
    rr_hosp = risk_ratio(ct_hosp.iloc[1, 1], ct_hosp.iloc[1].sum(), ct_hosp.iloc[0, 1], ct_hosp.iloc[0].sum())
    print(f"RR among hospitalized = {rr_hosp:.3f}")
    print(f"Whole-sample RR = {rr_all:.3f}")
    print(f"\n→ The RR changed after restricting to hospitalized patients!")
    print("→ This is collider bias")

print("\n=== Explaining Collider Bias ===")
print("hospitalized ← severity ← infection")
print("hospitalized ← infection")
print("→ hospitalized is a collider, jointly affected by severity and infection")
print("→ Conditioning on the collider (looking only at hospitalized patients) = opening a spurious path")
print("→ Result: among hospitalized patients, the relationship between shower_use and infection is distorted")

### Interpretation

- **AR/PAR**: the attributable risk differs between hydrotherapy and showering, reflecting the contributions of different exposure routes. Showering is the main route that generates Legionella aerosols
- **DiD intervention date**: the result is sensitive to the intervention date. The correct approach is to use the actual intervention date, not to pick the most significant one after the fact
- **Collider**: analyzing only hospitalized patients = conditioning on a collider, which introduces selection bias. This is a common trap in observational studies
- **Limits of causal inference**: with observational data, we can never be fully certain about causation. DAGs and statistical methods can only help us identify and reduce bias—they cannot eliminate all unobserved confounders

## Question 4 Solution

In [ ]:
# Vaccination policy DiD: some districts rolled out a vaccination booster campaign (treated); compare incidence before and after the policy
rng = np.random.default_rng(1204)
_rows = []
TRUE_EFFECT = -8.0   # the policy truly reduces incidence by 8/100k
for dz in range(200):
    treated = 1 if dz < 100 else 0
    base = rng.normal(45, 6)
    for post in (0, 1):
        inc = base - 3 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 4)
        _rows.append({"district": dz, "treated": treated, "post": post, "incidence": inc})
vax = pd.DataFrame(_rows)
print(f"DiD data: {vax['district'].nunique()} districts × 2 periods, true effect = {TRUE_EFFECT}/100k")

m = vax.groupby(["treated", "post"])["incidence"].mean().unstack()
did_manual = (m.loc[1, 1] - m.loc[1, 0]) - (m.loc[0, 1] - m.loc[0, 0])
fit = smf.ols("incidence ~ treated * post", vax).fit()
print(m.round(2))
print(f"\nManual DiD = {did_manual:.2f}")
print(f"Regression interaction term treated:post = {fit.params['treated:post']:.2f} "
      f"(95% CI {fit.conf_int().loc['treated:post', 0]:.2f} ~ {fit.conf_int().loc['treated:post', 1]:.2f})")
print("Interpretation: the interaction term is close to the true value -8; DiD uses the control group's before-after change to represent 'the common trend that would have occurred without the policy', and subtracting it out yields the policy's net effect.")

## Question 5 Solution

In [ ]:
# Mask mandate DiD: some counties implemented a mask mandate (treated); outcome is the weekly case growth rate
rng = np.random.default_rng(1205)
_rows = []
TRUE_EFFECT = -0.18   # the mask mandate reduces the growth rate by 0.18
for reg in range(160):
    treated = 1 if reg < 80 else 0
    base = rng.normal(0.30, 0.05)
    for post in (0, 1):
        g = base - 0.05 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 0.04)
        _rows.append({"region": reg, "treated": treated, "post": post, "growth": g})
mask = pd.DataFrame(_rows)
print(f"DiD data: {mask['region'].nunique()} counties × 2 periods, true effect = {TRUE_EFFECT}")

fit = smf.ols("growth ~ treated * post", mask).fit()
did = fit.params["treated:post"]
print(f"DiD (mask mandate effect) = {did:.3f} "
      f"(95% CI {fit.conf_int().loc['treated:post', 0]:.3f} ~ {fit.conf_int().loc['treated:post', 1]:.3f})")
print("Interpretation: the interaction term is negative and close to the true value -0.18 → the mask mandate reduces the weekly case growth rate by about 0.18.")

## Question 6 Solution

In [ ]:
# Smoking and disease: age is a confounder (older people smoke more often and are also more likely to get sick)
rng = np.random.default_rng(1206)
n = 3000
age = rng.integers(20, 80, n)
smoke = rng.binomial(1, 1 / (1 + np.exp(-(-2.5 + 0.05 * age))))
logit = -4.5 + 0.05 * age + 0.8 * smoke     # smoke's true log-OR = 0.8
disease = rng.binomial(1, 1 / (1 + np.exp(-logit)))
dat = pd.DataFrame({"age": age, "smoke": smoke, "disease": disease})
print(f"n={n}, smoking rate={smoke.mean():.1%}, disease rate={disease.mean():.1%} (smoke's true log-OR=0.8)")

crude = smf.logit("disease ~ smoke", data=dat).fit(disp=0)
adj = smf.logit("disease ~ smoke + age", data=dat).fit(disp=0)
print(f"crude    smoke coefficient (log-OR) = {crude.params['smoke']:.3f}  → OR={np.exp(crude.params['smoke']):.2f}")
print(f"adjusted smoke coefficient (log-OR) = {adj.params['smoke']:.3f}  → OR={np.exp(adj.params['smoke']):.2f}  (true value 0.8)")
print("Interpretation: the crude estimate is overestimated (age raises both smoking and disease, positive confounding); after adjusting for age, the smoke coefficient returns close to the true value.")

## Question 7 Solution

In [ ]:
# Propensity-score matching for COVID-19 treatment: sicker patients are more likely to be treated (confounding), but treatment is actually beneficial
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
rng = np.random.default_rng(1207)
n = 2000
severity = rng.uniform(0, 1, n)
treated = rng.binomial(1, 0.15 + 0.7 * severity)  # sicker patients are treated more often (confounding by indication; overlap is preserved)
TRUE_EFFECT = -0.15                              # treatment truly reduces the death rate by 0.15
death_p = (0.10 + 0.75 * severity + TRUE_EFFECT * treated).clip(0.01, 0.99)
death = rng.binomial(1, death_p)
cov = pd.DataFrame({"severity": severity, "treated": treated, "death": death})
print(f"n={n}, treated proportion={treated.mean():.1%}, true treatment effect (death-rate difference)={TRUE_EFFECT}")

naive = cov.loc[cov.treated == 1, "death"].mean() - cov.loc[cov.treated == 0, "death"].mean()
ps = LogisticRegression(max_iter=1000).fit(cov[["severity"]], cov["treated"]).predict_proba(cov[["severity"]])[:, 1]
cov["ps"] = ps
tr = cov[cov.treated == 1]; ct = cov[cov.treated == 0]
nn = NearestNeighbors(n_neighbors=1).fit(ct[["ps"]].values)
_, idx = nn.kneighbors(tr[["ps"]].values)
matched_ctrl_death = ct["death"].values[idx.ravel()]
att = tr["death"].mean() - matched_ctrl_death.mean()
print(f"naive death-rate difference = {naive:+.3f} (biased: sicker patients are treated more often, so treatment looks harmful)")
print(f"propensity-score-matched ATT = {att:+.3f} (true value {-0.15})")
print("Interpretation: matching makes the severity distributions of the treated and control groups similar; once confounding by indication is removed, the protective effect of treatment becomes apparent.")

## Question 8 Solution

In [ ]:
# Instrumental variables IV: the exposure is affected by an unobserved confounder U (endogenous); Z is the instrumental variable (Challenge)
rng = np.random.default_rng(1208)
n = 3000
U = rng.normal(0, 1, n)                 # unobserved confounder
Z = rng.normal(0, 1, n)                 # instrumental variable: affects exposure, does not directly affect the outcome
exposure = 0.6 * Z + 0.7 * U + rng.normal(0, 1, n)
TRUE_EFFECT = 1.5
outcome = TRUE_EFFECT * exposure + 1.2 * U + rng.normal(0, 1, n)
iv = pd.DataFrame({"Z": Z, "exposure": exposure, "outcome": outcome})
print(f"n={n}, true causal effect of exposure = {TRUE_EFFECT} (naive OLS will overestimate this due to U)")

naive = smf.ols("outcome ~ exposure", iv).fit()
stage1 = smf.ols("exposure ~ Z", iv).fit()
iv["exp_hat"] = stage1.fittedvalues
stage2 = smf.ols("outcome ~ exp_hat", iv).fit()
print(f"naive OLS coefficient = {naive.params['exposure']:.3f} (overestimated due to unobserved confounder U)")
print(f"IV (2SLS) coefficient = {stage2.params['exp_hat']:.3f} (true value 1.5)")
print(f"first-stage Z→exposure coefficient = {stage1.params['Z']:.3f}, F≈{stage1.fvalue:.0f} (instrument is strong enough)")
print("Interpretation: IV uses exogenous variation (Z) that affects the outcome only through the exposure to estimate the causal effect;")
print("Z must satisfy: (1) relevance (correlated with the exposure), (2) exclusion restriction (affects the outcome only through the exposure), and (3) independence from U.")

## Question 9: Food Poisoning AR/PAR and DiD for a Sanitation Intervention

A suspected food poisoning outbreak occurred in an elementary school's lunch program (this problem's data is a synthetic teaching scenario, not a real case). Health authorities suspect the "cold cucumber salad" served that day as the likely contamination source; after an audit, the catering company was required to strengthen its sanitation and disinfection measures.

**Part 1: AR / PAR (Attributable Risk)**

1. Use the `a, b, c, d` values from the 2×2 table below to compute the attack rates for the "ate the cold cucumber salad" and "did not eat it" groups
2. Compute the risk ratio (RR) and attributable risk (AR, risk difference)
3. Use the Levin formula to compute the population attributable fraction (PAF): `Pe * (RR - 1) / (1 + Pe * (RR - 1))`, where Pe is the proportion of everyone who ate that dish
4. Interpret the PAF: if this dish were removed from the menu, what proportion of cases could theoretically be prevented?

**Part 2: DiD (Effect of the Sanitation Measures)**

5. Using the panel data below, estimate the intervention effect of the sanitation measures with `smf.ols("cases ~ treated + post + treated:post", data=food).fit(cov_type="HC3")` (HC3-robust standard errors)
6. What does the `treated:post` interaction represent? How does it compare with the true effect built into the data?
7. What is the key assumption needed for DiD to hold (the parallel trends assumption)? If the case trends for the treated and control groups were not already parallel before the intervention, how would that distort the result?

In [ ]:
# Food poisoning 2x2 table: ate the cold cucumber salad vs did not x became ill vs did not (synthetic teaching data, not a real case)
a, b = 90, 30    # ate the suspect dish: ill / not ill
c, d = 20, 180   # did not eat the suspect dish: ill / not ill
print(f"Ate the suspect dish: {a + b} people (ill {a}, not ill {b})")
print(f"Did not eat the suspect dish: {c + d} people (ill {c}, not ill {d})")

# Sanitation-measure DiD panel: some schools received enhanced sanitation (treated); compare daily reported case counts before vs after the intervention (synthetic teaching data)
rng = np.random.default_rng(1209)
_rows = []
TRUE_EFFECT = -4.0   # enhanced sanitation measures reduce average daily reported cases by 4
for school in range(120):
    treated = 1 if school < 60 else 0
    base = rng.normal(10, 2)
    for post in (0, 1):
        cases = base - 1 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 1.5)
        _rows.append({"school": school, "treated": treated, "post": post, "cases": cases})
food = pd.DataFrame(_rows)
print(f"DiD data: {food['school'].nunique()} schools x 2 periods, true effect = {TRUE_EFFECT} cases/day")

In [ ]:
# Food poisoning AR/PAR: ate the cold cucumber salad vs did not
risk_exp = a / (a + b)
risk_unexp = c / (c + d)
RR = risk_exp / risk_unexp
AR = risk_exp - risk_unexp
Pe = (a + b) / (a + b + c + d)
PAF = Pe * (RR - 1) / (1 + Pe * (RR - 1))

print("=== Cold cucumber salad exposure: AR / PAR ===")
print(f"Attack rate among those who ate the suspect dish: {risk_exp:.1%}")
print(f"Attack rate among those who did not: {risk_unexp:.1%}")
print(f"Risk ratio RR = {RR:.2f}")
print(f"Attributable risk AR = {AR:.3f}")
print(f"Pe (proportion of everyone who ate the dish) = {Pe:.1%}")
print(f"Population attributable fraction PAF (Levin formula) = {PAF:.1%}")
print(f"→ If this dish were completely removed, about {PAF:.0%} of cases could theoretically be prevented")

# Sanitation-measure DiD
fit = smf.ols("cases ~ treated + post + treated:post", data=food).fit(cov_type="HC3")
did_coef = fit.params["treated:post"]
did_p = fit.pvalues["treated:post"]
ci_lo, ci_hi = fit.conf_int().loc["treated:post"]
print("\n=== Sanitation-measure DiD ===")
print(f"treated:post coefficient = {did_coef:.2f} (95% CI {ci_lo:.2f} to {ci_hi:.2f}, p = {did_p:.4f})")
print(f"True effect = {TRUE_EFFECT}")
print("→ The interaction is negative and close to the true value, showing daily reported cases dropped significantly after the enhanced sanitation measures")
print("→ Parallel trends assumption: DiD requires that, absent the intervention, the treated and control groups' case trends would have stayed parallel;")
print("   if the two groups already had different trends before the intervention, DiD would mistake that trend difference for the intervention effect")